In [1]:
from torch.utils.data import Dataset, DataLoader
import numpy as np 
import torch 
import os 
data_path = os.path.join('data','UBFC-RPPG-Dataset')
subjects = os.listdir(data_path)  

In [2]:
sample_subject = subjects[0]
sample_roi = np.loadtxt(os.path.join(data_path, sample_subject, 'roi_colors.txt'), delimiter=',')
sample_roi.shape 

(2035, 9)

In [3]:
sample_signal = np.loadtxt(os.path.join(data_path, sample_subject, 'ground_truth.txt'))


In [4]:
#each frame has 3 labels in the dataset : Normalized Blood Volume Pulse , Heart Rate and Time. We will only train on the first one


class UBFC_Dataset(Dataset):
    def __init__(self, data_path, subjects,seq_len=10):
        self.data_path = data_path 
        self.subjects = subjects 
        self.seq_len = seq_len
        self.possible_ranges = []
        for subject in subjects:
            signal_path = os.path.join(data_path,subject, 'ground_truth.txt')
            signal = np.loadtxt(signal_path)
            num_starts = signal.shape[-1] - seq_len 
            for i in range(num_starts): 
                self.possible_ranges.append((subject, i)) 
    def __len__(self):
        return len(self.possible_ranges)

    def __getitem__(self, index):
        subject,i = self.possible_ranges[index]
        signal_path = os.path.join(self.data_path,subject, 'ground_truth.txt')
        colors_path = os.path.join(self.data_path, subject, 'roi_colors.txt')
        signals = np.loadtxt(signal_path)
        colors = np.loadtxt(colors_path, delimiter=',')
        signal_seq = signals[0,i : i + self.seq_len]
        color_seq = colors[i : i + self.seq_len]
        return torch.tensor(color_seq, dtype=torch.float32), torch.tensor(signal_seq, dtype=torch.float32)
        

        




In [7]:
batch_size = 32
seq_len = 128
dataset = UBFC_Dataset(data_path, subjects,seq_len=seq_len)

from  torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size 
train_dataset, test_dataset = random_split(dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size = batch_size, shuffle=False)